<figure>
  <IMG SRC="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Fachhochschule_Südwestfalen_20xx_logo.svg/320px-Fachhochschule_Südwestfalen_20xx_logo.svg.png" WIDTH=250 ALIGN="right">
</figure>

# Einführung Machine Learning
### Sommersemester 2026
Prof. Dr. Heiner Giefers


# Naive Bayes Textklassifikation: Spamfilter

In dieser Aufgabe implementieren und evaluieren Sie einen einfachen **Naive Bayes Klassifikator für Textdaten**. Ziel ist es, auf Basis einer gegebenen CSV-Datei `email.csv`, die deutsche E-Mails sowie eine Spam-Markierung enthält, einen Spamfilter zu bauen.

Die Daten liegen bereits vorverarbeitet vor: Jede Zeile enthält den Text einer E-Mail und eine Zielvariable:
- `Spam = 1`: Spam
- `Spam = 0`: Kein Spam

Sie führen folgende Schritte durch:
1. Tokenisierung und Vorbereitung der Trainingsdaten
2. Schätzung der Wahrscheinlichkeiten für Klassen und Wörter
3. Implementierung des Naive-Bayes-Prädiktors
4. Evaluation auf Testdaten


In [ ]:

import pandas as pd
import re
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split


df = pd.read_csv("https://raw.githubusercontent.com/fhswf/datasets/refs/heads/main/email.csv")



## Schritt 1: Tokenisierung

Implementieren Sie eine Funktion `tokenize`, die aus einem E-Mail-Text eine Liste von Wörtern extrahiert. Alle Wörter sollen in Kleinbuchstaben umgewandelt werden. Verwenden Sie reguläre Ausdrücke.

Beispiel:

```python
tokenize("Das ist ein Test.") ➞ ["das", "ist", "ein", "test"]
```


Da wir nicht noch groß- und kleingeschriebene Wörter unterscheiden wollen, wandeln wir den gesamten Text in Kleinbuchstaben um.
Die Methode `lower()`, die für Zeichenketten in Python definiert ist, macht die Umwandlung sehr einfach.

Nun brauchen wir noch einen Weg, die einzelnen Wörter aus dem Text zu extrahieren. Die String-Methode `split(' ')` teilt einen String nach Leerzeichen auf. Leider werden wir so die Sonderzeichen aus dem Text nicht los.

Besser ist die Methode `re.findall(...)` aus dem Modul `re` (*Regular Expression*), um alle Wörter zu extrahieren.

Ein regulärer Ausdruck ist eine Zeichenkette, die ein Muster beschreibt. Du kannst dir dabei folgende Fragen stellen:

- Was ist ein **Wort**? (Tipp: Ein zusammenhängender Block von Buchstaben oder Buchstaben + Zahlen)
- Wie schreibt man „**mindestens ein Buchstabe**“ als regulären Ausdruck?
- Wie erkennt man **Wortgrenzen**, also Anfang oder Ende eines Wortes?

**Hinweise zur Konstruktion des Regulären Ausdrucks:**

- In regulären Ausdrücken steht `\w` für ein sogenanntes „word character“.
- Der Operator `+` bedeutet: *eins oder mehr* von diesen Zeichen.
- `\b` steht für eine Wortgrenze (beginnt oder endet ein Wort).

**Aufgabe:** Bauen Sie aus diesen Bausteinen einen Ausdruck, der Wörter im Text erkennt. Man kann den Ausdruck jederzeit in `re.findall(...)` testen.



In [ ]:


def tokenize(text):
    regex = r"\w"
    text_klein = "Hallo Welt!"
    # 1. Setze regex auf die Korrekt Regular Expression
    # 2. Berechne text_klein aus text, sodass text_klein nur Kleinbuchstaben enthält
    # YOUR CODE HERE
    raise NotImplementedError()
    return re.findall(regex, text_klein)



In [ ]:
# Test für tokenize
assert tokenize("Das ist ein Test!") == ["das", "ist", "ein", "test"]
print("Tokenisierung OK")


## Schritt 2: Trainingsdaten vorbereiten

Wir nutzen nun die `tokenize`-Funktion, um eine neue Spalte `tokens` im DataFrame zu erzeugen.

**Aufgabe:** Teilen Sie die Daten anschließend mit `train_test_split` in Trainings- und Testdaten auf (80%/20%).


In [ ]:
df["tokens"] = df["Text"].apply(tokenize)
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert "tokens" in df.columns
assert len(train_data) + len(test_data) == len(df)
print("Split und Tokenisierung OK")


## Schritt 3: Wahrscheinlichkeiten schätzen

In diesem Abschnitt wird das Training eines Naive-Bayes-Klassifikators vorbereitet. Ziel ist es, **Wortwahrscheinlichkeiten für jede Klasse (Spam / Nicht-Spam)** zu berechnen.

Hier erklären wir den Code Schritt für Schritt:

---

#### 1. Klassenhäufigkeiten zählen

```python
class_counts = train_data["Spam"].value_counts().to_dict()
total_docs = len(train_data)
```

- Zählt, wie viele E-Mails es pro Klasse gibt (`0` = Nicht-Spam, `1` = Spam)
- `total_docs` ist die Gesamtzahl der Trainingsdokumente

#### 2. A-priori-Wahrscheinlichkeiten berechnen

```python
class_probs = {c: count / total_docs for c, count in class_counts.items()}
```

- Berechnet für jede Klasse die Wahrscheinlichkeit $P(\text{Klasse})$
- Beispiel: Wenn 30% der Trainingsdaten Spam sind $\rightarrow$ `class_probs[1] = 0.3`

#### 3. Vorbereitungen für Wortzählung

```python
word_counts = {0: defaultdict(int), 1: defaultdict(int)}
class_word_totals = {0: 0, 1: 0}
```

- `word_counts`: Wörter + Häufigkeit für jede Klasse (als Wörterbuch)



In [ ]:
class_counts = train_data["Spam"].value_counts().to_dict()
total_docs = len(train_data)
class_probs = {c: count / total_docs for c, count in class_counts.items()}

word_counts = {0: defaultdict(int), 1: defaultdict(int)}
class_word_totals = {0: 0, 1: 0}
vocabulary = []

#### 4. Schleife über alle Trainings-E-Mails

```python
for _, row in train_data.iterrows():
    label = row["Spam"]
    for word in row["tokens"]:
        word_counts[label][word] += 1
        class_word_totals[label] += 1
        vocabulary.add(word)
```

- Für jede E-Mail:
  - Schaue dir das Label (0 oder 1) an
  - Für jedes Wort:
    - Zähle es bei der passenden Klasse
    - Erhöhe die Gesamtwortzahl dieser Klasse
    - Füge es zum Wortschatz (`vocabulary`) hinzu


In [ ]:
for _, row in train_data.iterrows():
    label = row["Spam"]
    for word in row["tokens"]:
        word_counts[label][word] += 1
        class_word_totals[label] += 1
        vocabulary.append(word)

#### 5. Berechne die Größe des Vokabulars

**Aufgabe:**  Erstellen Sie eine Variable `vocab_size`, die die *Anzahl der eindeutigen Wörter* im Vokabular enthält. Bisher enthält das Vokabular noch viele Wörter doppelt. Mit einem einfachen Trick können wir aus der Liste `vocabulary` eine Menge (`set`) machen. Wie in der Mathematik hat auch in Python die Datenstruktur Menge (`set`) die Eingenschaft, dass alle Elemente nur einmalig vorkommen.

In [ ]:
vocab_size = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert isinstance(vocabulary, set), "'vocabulary' sollte ein Set sein."
# Ist das Vokabular nicht leer?
assert len(vocabulary) > 0, "Das 'vocabulary' sollte Wörter enthalten (nicht leer sein)."


## Schritt 4: Klassifikator implementieren

Mit der Funktion `predict(text)` können wir nun neue E-Mail klassifizieren, also vorhersagen, ob sie **Spam (1)** oder **Nicht-Spam (0)** ist.

Dabei wird das Naive-Bayes-Prinzip genutzt: Für jede Klasse wird berechnet, wie wahrscheinlich es ist, dass der Text zu dieser Klasse gehört. Die Klasse mit der höchsten Wahrscheinlichkeit wird zurückgegeben.

---

#### 1. Text in Wörter zerlegen

```python
tokens = tokenize(text)
```

Die Funktion `tokenize` zerlegt den eingegebenen Text in einzelne Wörter. Groß-/Kleinschreibung wird dabei ignoriert.

#### 2. Wahrscheinlichkeiten für beide Klassen berechnen

```python
for label in [0, 1]:
    log_prob = np.log(class_probs[label])
```
Es wird für jede Klasse (0 = Nicht-Spam, 1 = Spam) ein Score berechnet. Gestartet wird mit dem Logarithmus der A-priori-Wahrscheinlichkeit:
$$
\log(P(\text{Spam})) \quad \text{bzw.} \quad \log(P(\text{Nicht-Spam}))
$$

**Achtung, Hinweis:** Warum wenden wir hier den Logarithmus an? Weil bei vielen Wörtern sehr kleine Wahrscheinlichkeiten miteinander multipliziert werden müssten — das führt schnell zu Zahlen, die der Computer nicht mehr exakt darstellen kann. Mit Logarithmen werden diese Produkte zu Summen.

#### 3. Wörter des Textes einrechnen (Likelihoods)

```python
for word in tokens:
    count = word_counts[label][word]
    prob = (count + 1) / (class_word_totals[label] + vocab_size)
    log_prob += np.log(prob)
```

- Für jedes Wort im Text:
  - Schaue nach, wie oft es in der Trainingsmenge der aktuellen Klasse vorkam.
  - Wende Laplace-Glättung an: $+1$ im Zähler, Vokabulargröße im Nenner.
  - Berechne die Log-Wahrscheinlichkeit und addiere sie zum `log_prob`.
- Das ergibt die logarithmierte Wahrscheinlichkeit, dass dieser Text zur Klasse `label` gehört.

In [ ]:
def predict(text):
    tokens = tokenize(text)
    scores = {}
    for label in [0, 1]:
        log_prob = np.log(class_probs[label])
        for word in tokens:
            count = word_counts[label][word]
            prob = (count + 1) / (class_word_totals[label] + vocab_size)
            log_prob += np.log(prob)
        scores[label] = log_prob
    return max(scores, key=scores.get)

In [ ]:
# Kleiner Test
assert predict("Liebe Kollegen") == 0
assert predict("Glückwunsch! Sie haben Gewonnen!") == 1
print("Klassifikator OK")


## Schritt 5: Evaluation

Wir wenden nun die Funktion auf das Testset an und berechnen Sie die Genauigkeit (Accuracy).


In [ ]:
test_data["predicted"] = test_data["Text"].apply(predict)
accuracy = (test_data["predicted"] == test_data["Spam"]).mean()
print(f"Accuracy auf Testdaten: {accuracy:.2%}")

## Wie löst man die Aufgabe mit scikit-learn

Um einen Spam-Klassifikator mit Naive Bayes in `scikit-learn` zu bauen, geht man in mehreren Schritten vor. Der größte Vorteil: Viele der aufwendigen Teilschritte wie Tokenisierung, Merkmalsextraktion und Wahrscheinlichkeitsberechnung werden vom Framework übernommen.

Wie oben laden wir die Daten in einen DataFrame und teilen in Trainings- und Testdaten auf

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

df = pd.read_csv("https://raw.githubusercontent.com/fhswf/datasets/refs/heads/main/email.csv")
X_train, X_test, y_train, y_test = train_test_split(df["Text"], df["Spam"], test_size=0.2, random_state=42)

Nun müssen wir unser **Bag-of-Words-Modell** erstellen, also den Text in einen Zahlen-Vektor umwandeln.
Hierzu verwenden wir die Klasse `CountVectorizer`, mit der man ein Bag-of-Words-Modell erstellen kann.

In [ ]:
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

**Aufgabe:** Trainieren Sie nun ein `MultinomialNB` Modell mit den Bag-of-Words-Modell `X_train_vec` und den Labels `y_train`.

Der `MultinomialNB`-Klassifikator in scikit-learn ist speziell für diskrete Merkmale, wie sie z. B. bei Wortzählungen im Bag-of-Words-Modell auftreten.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**Aufgabe:** Wenden Sie nun die `predict`-Methode, um die Vorhersagen für die Test-Vektoren `X_test_vec` zu berechnen. Bestimmen Sie dann mit der Methode `accuracy_score` die Accuracy `acc`.

In [ ]:
acc = None
# YOUR CODE HERE
raise NotImplementedError()
print(f"Genauigkeit: {acc:.2%}")